# Baseline Models: TF-IDF + SVM Pipeline

This notebook implements the non-Deep Learning solution proposed for the ASHO-AI ICD-10 codification task.

**Pipeline overview:**
1. **Preprocessing (Normalisation):** Lowercase, strip accents, remove punctuation/digits.
2. **Feature Extraction (TF-IDF):** Character + word n-grams with TF-IDF weighting.
3. **Classification (SVM):** Linear SVM wrapped in OneVsRest for multi-label.
4. **Prediction:** Generate leaderboard submission.

We additionally train Logistic Regression and Multinomial Naive Bayes baselines to contextualise the SVM results.

## 1. Setup

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make the src module importable from the notebooks folder
sys.path.insert(0, os.path.abspath('../src'))

from data_processing import (
    normalize_text, normalize_texts,
    prepare_multilabel_dataset, split_and_binarize
)
from evaluation import (
    calculate_metrics, print_comparison_table,
    plot_comparison, generate_submission
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

print('Setup complete.')

## 2. Data Loading

In [ ]:
codif_df = pd.read_csv('../data/codification_data.csv')
lead_df  = pd.read_csv('../data/leaderboard_data.csv')

print(f'Training rows : {len(codif_df):,}')
print(f'Unique codes  : {codif_df["Code"].nunique():,}')
print(f'Unique literals: {codif_df["Literal"].nunique():,}')
print(f'Leaderboard   : {len(lead_df):,}')
print()
display(codif_df.head())
display(lead_df.head())

## 3. Phase 1 — Preprocessing (Normalisation)

Following the proposal:
- **Lowercasing**
- **Accent stripping** (á→a, ñ→n, etc.)
- **Punctuation & digit handling**
- **Whitespace trimming**

We use `normalize_text()` from `src/data_processing.py` which implements all four steps.

In [ ]:
# Show normalisation effect on sample texts
samples = [
    'Hiperreactividad bronquial',
    'broncoesp\u00e1stica',
    'miocardiopat\u00eda dilatada',
    'Crisis febriles at\u00edpicas',
    'pr\u00f3tesis mama',
    'ALERGIA IBUPROFENO',
    '<font>VHC</font>',
    'H\u00e8rnia ventral',
]

print(f'{"Original":<35} → {"Normalised"}')
print('-' * 65)
for s in samples:
    print(f'{s:<35} → {normalize_text(s)}')

## 4. Dataset Preparation (Multi-label)

Many literals map to more than one ICD code. We group by literal, apply a minimum code-frequency filter (to remove extremely rare codes that would be nearly impossible to predict), then binarise the labels for multi-label classification.

In [ ]:
# Prepare multi-label dataset
# min_code_freq=2: keep codes appearing at least twice (drop singletons)
MIN_CODE_FREQ = 2

X_all, y_all, kept_codes = prepare_multilabel_dataset(
    codif_df,
    literal_col='Literal',
    code_col='Code',
    min_code_freq=MIN_CODE_FREQ,
)

print(f'\nKept codes ({MIN_CODE_FREQ}+ occurrences): {len(kept_codes)}')
print(f'Total literals: {len(X_all)}')

In [ ]:
# Apply normalisation to all literals
X_all_norm = normalize_texts(X_all)

# Quick sanity check
for i in range(3):
    print(f'  {X_all[i]:<35} → {X_all_norm[i]}')
    print(f'    codes: {y_all[i]}')

In [ ]:
# Split into train / validation and binarise labels
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_val, y_train_bin, y_val_bin, mlb = split_and_binarize(
    X_all_norm, y_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f'Number of label classes: {len(mlb.classes_)}')

## 5. Phase 2 — Feature Extraction (TF-IDF)

We configure the TF-IDF vectoriser with:
- **Character n-grams** `(3, 6)`: Crucial for capturing typos, abbreviations, and morphological roots in Spanish medical text.
- **Sublinear TF**: Applies logarithmic TF scaling (`1 + log(tf)`) which dampens the effect of very frequent terms.
- **max_features**: Limits the vocabulary to avoid memory issues and overfitting.

In [ ]:
tfidf = TfidfVectorizer(
    analyzer='char_wb',        # character n-grams respecting word boundaries
    ngram_range=(3, 6),        # trigrams to 6-grams
    sublinear_tf=True,         # log-normalised TF
    max_features=100_000,      # keep top features
    min_df=2,                  # ignore very rare n-grams
    dtype=np.float32,          # save memory
)

print('Fitting TF-IDF on training data...')
t0 = time.time()
X_train_tfidf = tfidf.fit_transform(X_train)
t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')
print(f'  Feature matrix: {X_train_tfidf.shape}')
print(f'  Vocabulary size: {len(tfidf.vocabulary_):,}')

# Transform validation set (do NOT refit!)
X_val_tfidf = tfidf.transform(X_val)
print(f'  Val matrix: {X_val_tfidf.shape}')

## 6. Phase 3 — Classification

### 6.1 Primary model: Linear SVM (OneVsRest)

Linear SVMs are the standard choice for high-dimensional sparse TF-IDF features. We wrap `LinearSVC` in `OneVsRestClassifier` for multi-label support.

In [ ]:
print('Training LinearSVC (OneVsRest)...')
t0 = time.time()

svm_clf = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        max_iter=10_000,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    n_jobs=-1,
)
svm_clf.fit(X_train_tfidf, y_train_bin)

t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')

In [ ]:
# Evaluate SVM on validation
y_pred_svm = svm_clf.predict(X_val_tfidf)

# Get decision function scores for P@K
y_scores_svm = svm_clf.decision_function(X_val_tfidf)

svm_metrics = calculate_metrics(
    y_val_bin, y_pred_svm,
    y_pred_prob=y_scores_svm,
    k_values=[5, 8, 15],
)

print('\n--- SVM Results ---')
for k, v in svm_metrics.items():
    print(f'  {k:<20}: {v:.4f}')

### 6.2 Baseline: Logistic Regression (OneVsRest)

In [ ]:
print('Training Logistic Regression (OneVsRest)...')
t0 = time.time()

lr_clf = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver='lbfgs',
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    n_jobs=-1,
)
lr_clf.fit(X_train_tfidf, y_train_bin)

t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')

In [ ]:
y_pred_lr = lr_clf.predict(X_val_tfidf)
y_scores_lr = lr_clf.predict_proba(X_val_tfidf)

lr_metrics = calculate_metrics(
    y_val_bin, y_pred_lr,
    y_pred_prob=y_scores_lr,
    k_values=[5, 8, 15],
)

print('\n--- Logistic Regression Results ---')
for k, v in lr_metrics.items():
    print(f'  {k:<20}: {v:.4f}')

### 6.3 Baseline: Multinomial Naive Bayes (OneVsRest)

In [ ]:
print('Training Multinomial NB (OneVsRest)...')
t0 = time.time()

nb_clf = OneVsRestClassifier(
    MultinomialNB(alpha=0.1),
    n_jobs=-1,
)
nb_clf.fit(X_train_tfidf, y_train_bin)

t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')

In [ ]:
y_pred_nb = nb_clf.predict(X_val_tfidf)
y_scores_nb = nb_clf.predict_proba(X_val_tfidf)

nb_metrics = calculate_metrics(
    y_val_bin, y_pred_nb,
    y_pred_prob=y_scores_nb,
    k_values=[5, 8, 15],
)

print('\n--- Naive Bayes Results ---')
for k, v in nb_metrics.items():
    print(f'  {k:<20}: {v:.4f}')

## 7. Model Comparison

In [ ]:
results = {
    'TF-IDF + SVM': svm_metrics,
    'TF-IDF + LogReg': lr_metrics,
    'TF-IDF + NaiveBayes': nb_metrics,
}

comparison_df = print_comparison_table(results)
display(comparison_df)

In [ ]:
# Visual comparison
plot_comparison(results, save_path='../docs/plots/baseline_comparison.png')

## 8. Phase 4 — Leaderboard Prediction

We use the best-performing model (SVM) to predict ICD-10 codes for the leaderboard data.

**Important:** We apply the same normalisation and TF-IDF transformation (without refitting).

In [ ]:
# Normalise leaderboard literals
lead_literals_norm = normalize_texts(lead_df['Literal'].tolist())

# Transform with the ALREADY FITTED TF-IDF vectoriser
X_lead_tfidf = tfidf.transform(lead_literals_norm)
print(f'Leaderboard TF-IDF matrix: {X_lead_tfidf.shape}')

# Predict with SVM
y_lead_pred = svm_clf.predict(X_lead_tfidf)
print(f'Predictions shape: {y_lead_pred.shape}')

# Check how many literals got at least one prediction
if hasattr(y_lead_pred, 'toarray'):
    y_lead_dense = y_lead_pred.toarray()
else:
    y_lead_dense = np.array(y_lead_pred)

has_prediction = (y_lead_dense.sum(axis=1) > 0).sum()
print(f'Literals with ≥1 predicted code: {has_prediction:,} / {len(lead_df):,}')

In [ ]:
# For literals with zero predictions, fall back to the top decision-function code
y_lead_scores = svm_clf.decision_function(X_lead_tfidf)

no_pred_mask = y_lead_dense.sum(axis=1) == 0
n_empty = no_pred_mask.sum()
print(f'Literals with zero predictions (fallback needed): {n_empty}')

if n_empty > 0:
    # For those, set the top-1 class to 1
    empty_indices = np.where(no_pred_mask)[0]
    for idx in empty_indices:
        best_class = np.argmax(y_lead_scores[idx])
        y_lead_dense[idx, best_class] = 1
    print(f'  → Assigned fallback top-1 code to {n_empty} literals')

In [ ]:
# Generate submission CSV
os.makedirs('../submissions', exist_ok=True)
submission_df = generate_submission(
    lead_df,
    y_lead_dense,
    mlb,
    output_path='../submissions/svm_baseline.csv',
)

display(submission_df.head(10))

## 9. Summary

| Step | Detail |
|------|--------|
| **Preprocessing** | Lowercased, stripped accents, removed punctuation/digits, collapsed whitespace |
| **Features** | Character n-grams (3–6) with TF-IDF, 100k max features |
| **Primary model** | LinearSVC + OneVsRest (multi-label), class_weight='balanced' |
| **Baselines** | Logistic Regression, Multinomial Naive Bayes |
| **Evaluation** | Micro-F1, Macro-F1, Precision, Recall, P@K |
| **Output** | `submissions/svm_baseline.csv` |